# MScFE 610 Financial Econometrics — Group Work Project 1
### Group 17817 · Module 6 · Technical notebook

This notebook holds the code and output for every computational answer in the report. Each section
is headed by the question it answers. Group 17817 is an **odd** group, so Problem 3 uses
`FE-GWP1_model_selection_1.csv`. Figures are written to `figures/`, data are read from `data/`.

| Section | Question | Content |
|---|---|---|
| Setup | — | Libraries, master seed, plotting conventions |
| 1 | **1e** | Omitted-variable bias by Monte Carlo simulation |
| 2 | **2b, 2c** | Sensitivity of OLS to outliers; the high-leverage limit |
| 3 | **3a, 3b** | Model selection on dataset 1; consistency of BIC versus AIC |
| 4 | **4c–4h** | Interpretation of the log-wage equation |
| 5 | **5c** | Unit roots, Johansen cointegration tests, VECM |
| 6 | **6a, 6b** | AR(1) MA(∞) representation; AR(2) autocorrelations |

Questions that are purely analytical (1a–1d, 2a, 4a–4b, 5a–5b) are answered in the written report.

## Setup

In [ ]:
import itertools, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss, acf
from statsmodels.tsa.vector_ar.vecm import coint_johansen, VECM, select_coint_rank, select_order
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_ljungbox, linear_reset
from statsmodels.stats.outliers_influence import OLSInfluence, variance_inflation_factor

warnings.filterwarnings("ignore")
pd.set_option("display.width", 130, "display.float_format", lambda v: f"{v:9.4f}")
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False, "figure.autolayout": True})
BLUE, RED, GREEN, PURPLE, GREY = "#2A6FDB", "#D1495B", "#3C8D5A", "#8C6BB1", "#6B7280"

SEED = 17817          # the group number, used as the master seed so every result is reproducible
DATA = "data"         # input CSV files
FIG  = "figures"      # every chart is saved here for the report
os.makedirs(FIG, exist_ok=True)

def save_fig(name):
    """Save the current figure to figures/<name>.png at print resolution, then display it."""
    plt.savefig(f"{FIG}/{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

print("Master random seed:", SEED)

## Question 1e — Omitted-variable bias in a controlled simulation

Data-generating process, with every parameter known by construction:

$$Y_i = a + bX_i + cZ_i + e_i,\qquad a=1,\; b=2,\; c=1.5,$$

where $(X_i, Z_i)$ are drawn from a bivariate normal with unit variances and correlation
$\rho=0.7$, and $e_i \sim N(0,1)$ is drawn independently of both regressors. We then estimate the
model twice: once correctly, and once with $Z$ omitted.

Theory predicts that omitting $Z$ drives the estimator of $b$ to
$\; \operatorname{plim}\hat b_{\text{short}} = b + c\,\dfrac{\operatorname{Cov}(X,Z)}{\operatorname{Var}(X)} = 2 + 1.5(0.7) = 3.05.$

In [ ]:
# Known data-generating process: Y = a + bX + cZ + e, with X and Z correlated by construction
A_TRUE, B_TRUE, C_TRUE = 1.0, 2.0, 1.5        # true intercept and slopes
RHO, SX, SZ, SE = 0.7, 1.0, 1.0, 1.0          # corr(X, Z), sd(X), sd(Z), sd(e)

def draw(n, rng, rho=RHO):
    """One sample of size n. e is drawn independently of X and Z, as the question requires."""
    cov = [[SX**2, rho*SX*SZ], [rho*SX*SZ, SZ**2]]
    XZ = rng.multivariate_normal([0, 0], cov, size=n)
    X, Z = XZ[:, 0], XZ[:, 1]
    Y = A_TRUE + B_TRUE*X + C_TRUE*Z + rng.normal(0, SE, n)
    return X, Z, Y

# Omitting Z shifts the slope on X by c * Cov(X,Z)/Var(X) = c * rho * sd(Z)/sd(X)
theory_bias = C_TRUE * RHO * SZ / SX
print(f"True b = {B_TRUE};  theoretical asymptotic value of the misspecified estimator "
      f"= b + c*rho = {B_TRUE + theory_bias:.4f}")

In [ ]:
# --- 1e(i) one sample of 200 observations, estimated both ways -------------------------------
rng = np.random.default_rng(SEED)
X, Z, Y = draw(200, rng)
# Fit the correct model and the misspecified one on the same data
full  = sm.OLS(Y, sm.add_constant(np.column_stack([X, Z]))).fit()
short = sm.OLS(Y, sm.add_constant(X)).fit()
aux   = sm.OLS(Z, sm.add_constant(X)).fit()          # auxiliary regression of the omitted variable on X

print("Correct specification   Y ~ X + Z")
print(f"   a = {full.params[0]:.4f} (se {full.bse[0]:.4f})   b = {full.params[1]:.4f} (se {full.bse[1]:.4f})"
      f"   c = {full.params[2]:.4f} (se {full.bse[2]:.4f})   R2 = {full.rsquared:.4f}   s = {np.sqrt(full.mse_resid):.4f}")
print("Misspecified            Y ~ X          (Z omitted)")
print(f"   a = {short.params[0]:.4f} (se {short.bse[0]:.4f})   b = {short.params[1]:.4f} (se {short.bse[1]:.4f})"
      f"                        R2 = {short.rsquared:.4f}   s = {np.sqrt(short.mse_resid):.4f}")
print(f"\nAuxiliary regression Z on X: slope = {aux.params[1]:.4f}")
# The omitted-variable formula holds exactly in-sample, not only on average
print(f"Algebraic identity  b_short = b_full + c_full * slope(Z on X):"
      f"  {short.params[1]:.6f}  =  {full.params[1] + full.params[2]*aux.params[1]:.6f}")
print(f"\nCorrelation of the short-model residual with X = {np.corrcoef(short.resid, X)[0,1]:.2e}  (zero by construction)")
print(f"Correlation of the short-model residual with Z = {np.corrcoef(short.resid, Z)[0,1]:.4f}  "
      "(the omitted variable now lives inside the disturbance)")

In [ ]:
# --- 1e(ii) does a larger sample fix the problem?  5,000 replications at each sample size -----
NS, REP = [25, 50, 100, 200, 500, 1000, 5000, 20000], 5000
rows = []
for n in NS:
    rng = np.random.default_rng(SEED + n)                  # separate, reproducible stream per n
    bf, bs, sef, ses = [], [], [], []                      # slope and its s.e.: full / short model
    for _ in range(REP):
        X, Z, Y = draw(n, rng)
        f = sm.OLS(Y, sm.add_constant(np.column_stack([X, Z]))).fit()    # correct model
        s = sm.OLS(Y, sm.add_constant(X)).fit()                          # Z omitted
        bf.append(f.params[1]); bs.append(s.params[1]); sef.append(f.bse[1]); ses.append(s.bse[1])
    bf, bs, sef, ses = map(np.array, (bf, bs, sef, ses))
    # bias = mean error; RMSE combines bias and spread; cover95 = share of 95% intervals containing b
    rows.append(dict(n=n,
                     mean_correct=bf.mean(), sd_correct=bf.std(ddof=1), bias_correct=bf.mean()-B_TRUE,
                     rmse_correct=np.sqrt(((bf-B_TRUE)**2).mean()), cover95_correct=np.mean(np.abs(bf-B_TRUE) <= 1.96*sef),
                     mean_omitted=bs.mean(), sd_omitted=bs.std(ddof=1), bias_omitted=bs.mean()-B_TRUE,
                     rmse_omitted=np.sqrt(((bs-B_TRUE)**2).mean()), cover95_omitted=np.mean(np.abs(bs-B_TRUE) <= 1.96*ses)))
mc = pd.DataFrame(rows)
print(mc.to_string(index=False))
print(f"\nAsymptotic target for the misspecified estimator: {B_TRUE + theory_bias:.4f}")

In [ ]:
# --- 1e(iii) the bias tracks corr(X, Z) exactly ------------------------------------------------
# For each correlation, fit the short model (Z omitted) on 2,000 large samples and average the slope.
sens = []
for r in [-0.9, -0.5, -0.2, 0.0, 0.2, 0.5, 0.7, 0.9]:
    rng = np.random.default_rng(SEED + int(1000*r) + 7)
    bs = []
    for _ in range(2000):
        X, Z, Y = draw(5000, rng, rho=r)
        bs.append(sm.OLS(Y, sm.add_constant(X)).fit().params[1])
    sens.append(dict(corr_XZ=r, simulated_mean_b=np.mean(bs), predicted_b=B_TRUE + C_TRUE*r,
                     simulated_bias=np.mean(bs)-B_TRUE, predicted_bias=C_TRUE*r))
sens = pd.DataFrame(sens)
print(sens.to_string(index=False))

In [ ]:
# --- 1e figure ---------------------------------------------------------------------------------
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
# 3,000 samples at n = 200 give the sampling distribution of each estimator
rng = np.random.default_rng(SEED + 1)
bf_s, bs_s = [], []
for _ in range(3000):
    X, Z, Y = draw(200, rng)
    bf_s.append(sm.OLS(Y, sm.add_constant(np.column_stack([X, Z]))).fit().params[1])
    bs_s.append(sm.OLS(Y, sm.add_constant(X)).fit().params[1])

# (i) histogram of the two estimators against the true b and its biased limit
ax[0].hist(bf_s, 45, alpha=.75, color=BLUE, label="Both predictors included")
ax[0].hist(bs_s, 45, alpha=.75, color=RED,  label="Z omitted")
ax[0].axvline(B_TRUE, color="k", lw=1.6, label=f"True b = {B_TRUE}")
ax[0].axvline(B_TRUE+theory_bias, color=RED, ls="--", lw=1.6, label=f"b + c·ρ = {B_TRUE+theory_bias:.2f}")
ax[0].set(xlabel="Estimate of b", ylabel="Frequency out of 3,000 samples",
          title="(i) Sampling distributions at n = 200")
ax[0].legend(fontsize=7, frameon=False)

# (ii) mean +/- 1.96 SD across sample sizes, taken from the Monte Carlo table
ax[1].axhline(B_TRUE, color="k", lw=1.5, label=f"True b = {B_TRUE}")
ax[1].axhline(B_TRUE+theory_bias, color=RED, ls="--", lw=1.5, label=f"Asymptotic limit = {B_TRUE+theory_bias:.2f}")
ax[1].errorbar(mc.n, mc.mean_correct, yerr=1.96*mc.sd_correct, fmt="o-", color=BLUE, ms=4, capsize=2,
               label="Both predictors included")
ax[1].errorbar(mc.n, mc.mean_omitted, yerr=1.96*mc.sd_omitted, fmt="s-", color=RED, ms=4, capsize=2,
               label="Z omitted")
ax[1].set(xscale="log", xlabel="Sample size n (logarithmic scale)", ylabel="Mean estimate of b ± 1.96 SD",
          title="(ii) A larger sample does not remove the bias")
ax[1].legend(fontsize=7, frameon=False)

# (iii) simulated against predicted limit for each corr(X, Z)
ax[2].plot(sens.corr_XZ, sens.simulated_mean_b, "o", color=RED, ms=6, label="Simulated mean, Z omitted")
ax[2].plot(sens.corr_XZ, sens.predicted_b, "-", color="k", lw=1.4, label="Theory: b + c·corr(X, Z)")
ax[2].axhline(B_TRUE, color=GREY, ls=":", lw=1.4, label=f"True b = {B_TRUE}")
ax[2].set(xlabel="corr(X, Z)", ylabel="Mean estimate of b",
          title="(iii) Sign and size of the bias follow corr(X, Z)")
ax[2].legend(fontsize=7, frameon=False)
save_fig("fig_1e_omitted_variable_bias")

### Baseline replication — single draw per sample size (Answer.pdf)

The group's first pass at 1e used a different known model, $Y_i = 2 + 1.5X_i + 2Z_i + e_i$, with
$X_i = 0.7Z_i + \sqrt{1-0.7^2}\,u_i$ so that $\operatorname{corr}(X,Z)=0.7$, and one sample at each
size. Theory gives $\operatorname{plim}\hat b_{\text{short}} = 1.5 + 2(0.7) = 2.9$. The cell reproduces
the table in the written answer exactly (legacy seed 610).

In [ ]:
np.random.seed(610)          # legacy global seed, kept so the numbers match Answer.pdf exactly
rows = []
for n in [100, 500, 1000, 5000]:
    Zb = np.random.normal(0, 1, n)
    Xb = 0.7*Zb + np.sqrt(1 - 0.7**2)*np.random.normal(0, 1, n)   # unit variance, corr(X, Z) = 0.7
    Yb = 2 + 1.5*Xb + 2*Zb + np.random.normal(0, 1, n)            # e independent of X and Z
    # slope on X with Z included, then with Z omitted
    rows.append(dict(n=n,
                     b_with_X_and_Z=sm.OLS(Yb, sm.add_constant(np.column_stack([Xb, Zb]))).fit().params[1],
                     b_with_X_only=sm.OLS(Yb, sm.add_constant(Xb)).fit().params[1]))
print("True b = 1.5;  probability limit with Z omitted = 1.5 + 2(0.7) = 2.9\n")
print(pd.DataFrame(rows).to_string(index=False))

**Results.** With Z included, the estimate of $b$ is centred on the true value 2 at every sample size (bias below 0.004) and its spread shrinks like $1/\sqrt{n}$. With Z omitted, it is centred on $b + c\rho = 3.05$ at every sample size: the bias of about 1.05 is the same at $n = 25$ and at $n = 20{,}000$. Only the spread shrinks, so the 95% interval covers the true $b$ in 9% of samples at $n = 25$ and in none from $n = 200$ onwards. **A larger sample does not cure omitted-variable bias; it makes the wrong answer more precise.** The bias moves one-for-one with $c\cdot\operatorname{corr}(X,Z)$ and disappears only when $\operatorname{corr}(X,Z) = 0$. The baseline replication shows the same pattern with a different model: $\hat b \approx 2.8$–$2.9$ against a true 1.5, with no improvement as $n$ grows.

## Question 2b — How outliers move the fitted line

Construction of the simulation. Sixty clean observations are generated from
$y_i = 2 + 1.5x_i + \varepsilon_i$ with $x_i \sim U(0,10)$ and $\varepsilon_i \sim N(0,1)$. That clean
sample is then held **fixed** and contaminated in four different ways, so that every difference in the
estimates is attributable to the contamination alone and not to resampling noise:

| Scenario | Contamination | Design position |
|---|---|---|
| S0 | none — baseline | — |
| S1 | one point at $(5, 40)$ | central $x$, extreme $y$ → **vertical outlier** |
| S2 | one point at $(30, 47)$ | extreme $x$, *on* the true line → **good leverage** |
| S3 | one point at $(30, 5)$ | extreme $x$, off the line → **bad leverage** |
| S4 | three points near $(29, 5)$ | a cluster of bad-leverage points → **masking** |

Each contaminated sample is fitted by ordinary least squares and, for comparison, by Huber
M-estimation, which down-weights large residuals.

In [ ]:
rng = np.random.default_rng(SEED)
n0, a0, b0 = 60, 2.0, 1.5
x0 = rng.uniform(0, 10, n0)
y0 = a0 + b0*x0 + rng.normal(0, 1.0, n0)          # the clean sample, held fixed throughout

# Each scenario appends contaminating point(s) to the same clean sample
scen = {
    "S0 Clean baseline":                    (x0.copy(), y0.copy()),
    "S1 Vertical outlier (x=5, y=40)":      (np.append(x0, 5.0),  np.append(y0, 40.0)),
    "S2 Good leverage (x=30, on the line)": (np.append(x0, 30.0), np.append(y0, a0 + b0*30.0)),
    "S3 Bad leverage (x=30, y=5)":          (np.append(x0, 30.0), np.append(y0, 5.0)),
    "S4 Masking cluster (3 points)":        (np.concatenate([x0, [28., 29., 30.]]),
                                             np.concatenate([y0, [5., 4., 6.]])),
}

rows = []
for k, (xx, yy) in scen.items():
    Xd = sm.add_constant(xx)
    f = sm.OLS(yy, Xd).fit()
    inf = OLSInfluence(f)                                         # leverage and Cook's distance
    rlm = sm.RLM(yy, Xd, M=sm.robust.norms.HuberT()).fit()        # robust fit for comparison
    # leverage_cutoff = 2(k+1)/n, the usual flag for a high-leverage point (k = 1 slope)
    rows.append(dict(scenario=k, n=len(yy), intercept=f.params[0], slope=f.params[1],
                     se_slope=f.bse[1], t_slope=f.tvalues[1], R2=f.rsquared,
                     resid_sd=np.sqrt(f.mse_resid), max_leverage=inf.hat_matrix_diag.max(),
                     leverage_cutoff=4/len(yy), max_cooks_D=inf.cooks_distance[0].max(),
                     huber_slope=rlm.params[1]))
res2 = pd.DataFrame(rows)
res2["slope_change_vs_clean_%"] = 100*(res2.slope - res2.slope.iloc[0])/res2.slope.iloc[0]
print(f"True data-generating values: intercept = {a0}, slope = {b0}\n")
print(res2.to_string(index=False))

In [ ]:
# The same single point, moved further and further away in y, at two different x positions.
brk = []
for yout in [0, 10, 20, 40, 80, 160, 320]:
    brk.append(dict(y_of_contaminating_point=yout,
                    slope_when_x_is_5 =sm.OLS(np.append(y0, yout), sm.add_constant(np.append(x0, 5.0))).fit().params[1],
                    slope_when_x_is_30=sm.OLS(np.append(y0, yout), sm.add_constant(np.append(x0, 30.0))).fit().params[1]))
brk = pd.DataFrame(brk)
print(f"True slope = {b0}\n")
print(brk.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(13.5, 7.4)); axf = ax.ravel()
# One panel per scenario: clean points, contaminating points, true / OLS / Huber lines
grid = np.linspace(0, 32, 50)
for i, (k, (xx, yy)) in enumerate(scen.items()):
    a_ = axf[i]
    f = sm.OLS(yy, sm.add_constant(xx)).fit()
    r = sm.RLM(yy, sm.add_constant(xx), M=sm.robust.norms.HuberT()).fit()
    a_.scatter(x0, y0, s=14, color=BLUE, alpha=.65, label="Clean observations (n = 60)")
    extra = np.arange(n0, len(xx))
    if len(extra):
        a_.scatter(xx[extra], yy[extra], s=65, marker="D", color=RED, edgecolor="k", lw=.6,
                   zorder=5, label="Contaminating point(s)")
    a_.plot(grid, a0 + b0*grid, "k-", lw=1.3, label=f"True line (slope {b0})")
    a_.plot(grid, f.params[0] + f.params[1]*grid, "--", color=RED, lw=1.7,
            label=f"Least squares (slope {f.params[1]:.2f})")
    a_.plot(grid, r.params[0] + r.params[1]*grid, ":", color=GREEN, lw=1.9,
            label=f"Huber M-estimator (slope {r.params[1]:.2f})")
    a_.set(title=k, xlabel="x", ylabel="y", xlim=(-1, 32))
    a_.legend(fontsize=6.4, frameon=False, loc="upper left")

# Last panel: slope as the single contaminating point moves up, at low vs high leverage
axf[5].plot(brk.y_of_contaminating_point, brk.slope_when_x_is_5,  "o-", color=BLUE,
            label="Contaminating point at x = 5 (central)")
axf[5].plot(brk.y_of_contaminating_point, brk.slope_when_x_is_30, "s-", color=RED,
            label="Contaminating point at x = 30 (high leverage)")
axf[5].axhline(b0, color="k", lw=1.2, label=f"True slope = {b0}")
axf[5].set(xlabel="y-value of the single contaminating point", ylabel="Least-squares slope estimate",
           title="Leverage, not outlyingness, causes the damage")
axf[5].legend(fontsize=7, frameon=False)
save_fig("fig_2b_outliers")

### Baseline replication — three bad-leverage outliers (Answer.pdf)

The group's first pass at 2b drew 100 clean points from $Y_i = 2 + 3X_i + e_i$ with
$X_i, e_i \sim N(0,1)$, then added three points at $(8,-20), (9,-25), (10,-30)$: far out in $x$ and
far below the line. The cell reproduces the written answer exactly (legacy seed 610).

In [ ]:
np.random.seed(610)          # legacy global seed, kept so the numbers match Answer.pdf exactly
xb = np.random.normal(0, 1, 100)
yb = 2 + 3*xb + np.random.normal(0, 1, 100)                     # clean sample: true line y = 2 + 3x
xo, yo = np.array([8., 9., 10.]), np.array([-20., -25., -30.])  # far out in x AND far below the line
clean = sm.OLS(yb, sm.add_constant(xb)).fit()
cont  = sm.OLS(np.concatenate([yb, yo]), sm.add_constant(np.concatenate([xb, xo]))).fit()
# Leverage of the added points compared with the average leverage (k+1)/n
print(pd.DataFrame({"intercept": [clean.params[0], cont.params[0]], "slope": [clean.params[1], cont.params[1]]},
                   index=["without outliers (n = 100)", "with 3 outliers (n = 103)"]).to_string())
print(f"\nLeverage of the three added points: {np.round(OLSInfluence(cont).hat_matrix_diag[-3:], 3)}"
      f"  (average leverage = {2/103:.3f})")

**Results.** A vertical outlier at a central $x$ (S1) barely moves the slope (−2.6%), but it quadruples the residual standard deviation and cuts $R^2$ from 0.95 to 0.51. A high-leverage point that lies on the true line (S2) is harmless and even sharpens the slope estimate (standard error 0.031 against 0.047). A single high-leverage point off the line (S3) cuts the slope by 64%. A cluster of three (S4) cuts it by 89% and leaves it insignificant ($t = 1.81$), yet no single point looks as extreme as in S3 (largest Cook's distance 1.2 against 37), because the three points mask one another. The Huber estimator keeps the slope between 1.47 and 1.57 in S1–S3 but only partly resists the masking cluster (1.22). The sweep table shows the mechanism: moving one point in $y$ at $x = 5$ hardly changes the slope, while at $x = 30$ the slope follows the point. In the baseline replication, three points with leverage 0.18–0.28 (against an average of 0.019) turn a slope of 2.99 into −1.10.

## Question 2c — Numerical confirmation of the high-leverage limit

The report proves algebraically that when $n-1$ points lie on $y=0$ and a single point
$(x_n, y_n)$ is pushed out to $x_n \to \infty$, the least-squares slope satisfies
$\hat\beta \to y_n/x_n$ and the leverage of that point tends to one. The cell below checks both
claims numerically.

In [ ]:
# Question 2c set-up: n-1 = 49 points on the line y = 0 and one point (x_n, y_n = 7)
rng = np.random.default_rng(SEED)
nn = 50
xc, yc, yn = rng.uniform(0, 10, nn-1), np.zeros(nn-1), 7.0

# Push x_n outwards and compare the OLS slope with the ratio y_n / x_n
rows = []
for xn in [10, 1e2, 1e3, 1e4, 1e6, 1e8]:
    f = sm.OLS(np.append(yc, yn), sm.add_constant(np.append(xc, xn))).fit()
    rows.append(dict(x_n=xn, slope_hat=f.params[1], y_n_over_x_n=yn/xn,
                     ratio=f.params[1]/(yn/xn), intercept_hat=f.params[0],
                     leverage_of_outlier=OLSInfluence(f).hat_matrix_diag[-1]))   # h_nn -> 1
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:16.10g}"))
print("\nThe ratio slope_hat / (y_n / x_n) converges to 1 and the leverage converges to 1:")
print("the fitted line is pinned to the single far-out point and the other 49 observations are ignored.")

**Results.** As $x_n$ grows from $10^2$ to $10^8$, the ratio $\hat\beta / (y_n/x_n)$ falls from 1.013 to 1.00000006 and the leverage of the far point rises to 1. The fitted line passes through $(x_n, y_n)$, and its slope is set by $y_n/x_n$ alone, whatever the other 49 observations are.